# Analyze Alpine3D simulation using lateral flow on a synthetic DEM

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

### Define some general functions

In [ ]:
# Function to read ASCII grid file
def read_ascii_grid(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()

    # Parse header
    header = {}
    for line in lines:
        if line.strip().lower().startswith('ncols'):
            header['ncols'] = int(line.split()[1])
        elif line.strip().lower().startswith('nrows'):
            header['nrows'] = int(line.split()[1])
        elif line.strip().lower().startswith('xllcorner'):
            header['xllcorner'] = float(line.split()[1])
        elif line.strip().lower().startswith('yllcorner'):
            header['yllcorner'] = float(line.split()[1])
        elif line.strip().lower().startswith('cellsize'):
            header['cellsize'] = float(line.split()[1])
        elif line.strip().lower().startswith('nodata_value'):
            header['NODATA_value'] = float(line.split()[1])

    # Find the start of the data
    data_start = 0
    for i, line in enumerate(lines):
        if not line.strip().lower().startswith(('ncols', 'nrows', 'xllcorner', 'yllcorner', 'cellsize', 'nodata_value')):
            data_start = i
            break

    # Read data
    data = np.loadtxt(lines[data_start:])
    data = data.reshape((header['nrows'], header['ncols']))

    data = np.ma.masked_equal(data, header['NODATA_value']).filled(np.nan)
    return data, header

In [ ]:
# Set global font size for all text elements
plt.rcParams.update({
    'font.size': 16,          # Default font size for text
    'axes.titlesize': 16,     # Font size for titles
    'axes.labelsize': 16,     # Font size for x and y labels
    'xtick.labelsize': 16,    # Font size for x-axis tick labels
    'ytick.labelsize': 16,    # Font size for y-axis tick labels
    'legend.fontsize': 16,    # Font size for legend text
})

### Read model output and DEM

In [ ]:
ds_lateral = xr.open_dataset("../output/grids_lateral.nc")
ds_nolateral = xr.open_dataset("../output/grids_nolateral.nc")
ds_nolateral.close()
ds_lateral.close()

dem, dem_header = read_ascii_grid("../input/surface-grids/cone.asc")
dem_slope, dem_header = read_ascii_grid("../input/surface-grids/cone.slope")

print(ds_nolateral.time.values[0], ds_nolateral.time.values[-1])
print(ds_lateral.time.values[0], ds_lateral.time.values[-1])

# Runoff reduced
east1 = 782400
north1 = 176700

# Runoff increased
east2 = 783500
north2 = 178000

### Plot DEM and slope

In [ ]:
# Create the plot
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Plot the data
extent = (
    dem_header['xllcorner'],
    dem_header['xllcorner'] + dem_header['ncols'] * dem_header['cellsize'],
    dem_header['yllcorner'],
    dem_header['yllcorner'] + dem_header['nrows'] * dem_header['cellsize']
)

# Use imshow or pcolormesh
im = ax[0].imshow(dem, extent=extent, origin='lower', cmap='terrain')
im2 = ax[1].imshow(dem_slope, extent=extent, origin='lower', cmap='cividis')

# Add colorbar
cbar = plt.colorbar(im, ax=ax[0])
cbar.set_label('Elevation (m)')

# Set labels and title
ax[0].set_xlabel('Easting')
ax[0].set_ylabel('Northing')
ax[0].set_title('Synthetic DEM')

cbar = plt.colorbar(im2, ax=ax[1])
cbar.set_label('Slope angle (degrees)')

ax[1].set_xlabel('Easting')
ax[1].set_ylabel('Northing')
ax[1].set_title('Synthetic DEM')


ax[0].text(-0.15, 1.05, '(a)', transform=ax[0].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
ax[1].text(-0.15, 1.05, '(b)', transform=ax[1].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
#axes[2].text(-0.1, 1.1, 'c', transform=axes[2].transAxes, fontsize=14, fontweight='bold', va='top', ha='right')

plt.tight_layout()
plt.show()

### Plot difference in SNOWPACK runoff for a given time stamp

In [ ]:
# Choose time
t = "2012-04-12T17:00:00.000000000"

# Create side-by-side plots
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

# Plot first dataset
ds_nolateral["MS_SNOWPACK_RUNOFF"].sel(time=t).plot(ax=ax[0])
ax[0].set_title("MS_SNOWPACK_RUNOFF (ds)")

# Plot second dataset
ds_lateral["MS_SNOWPACK_RUNOFF"].sel(time=t).plot(ax=ax[1])
ax[1].set_title("MS_SNOWPACK_RUNOFF (ds_lateral)")

(ds_lateral["MS_SNOWPACK_RUNOFF"].sel(time=t)-ds_nolateral["MS_SNOWPACK_RUNOFF"].sel(time=t)).plot(ax=ax[2])
ax[1].set_title("MS_SNOWPACK_RUNOFF (ds_lateral)")


total_diff = (ds_lateral["MS_SNOWPACK_RUNOFF"].sel(time=t)-ds_nolateral["MS_SNOWPACK_RUNOFF"].sel(time=t)).sum(dim=("easting", "northing"))
print(total_diff.values)


plt.tight_layout()
plt.show()

### Plot difference in SNOWPACK snow depth for a given time stamp

In [ ]:
# Create side-by-side plots
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

# Plot first dataset
ds_nolateral["snd"].sel(time=t).plot(ax=ax[0])
ax[0].set_title("snd (ds_nolateral)")

# Plot second dataset
ds_lateral["snd"].sel(time=t).plot(ax=ax[1])
ax[1].set_title("snd (ds_lateral)")

(ds_lateral["snd"].sel(time=t)-ds_nolateral["snd"].sel(time=t)).plot(ax=ax[2])
ax[1].set_title("snd (ds_lateral - ds_nolateral)")

plt.tight_layout()
plt.show()

### Calculate runoff sums and average liquid water in snow between time stamps t1 and t2

In [ ]:
# Choose time
t1 = "2012-02-01T02:00:00.000000000"
#t1 = None
t2 = "2012-06-01T02:00:00.000000000"

runoff_nolateral_sum = {}
runoff_lateral_sum = {}

ms_water_nolateral_sum = {}
ms_water_lateral_sum = {}

runoff_nolateral_sum["MS_SNOWPACK_RUNOFF"] = (
    ds_nolateral["MS_SNOWPACK_RUNOFF"]
    .sel(time=slice(t1, t2))
    .sum(dim="time")
)

runoff_lateral_sum["MS_SNOWPACK_RUNOFF"] = (
    ds_lateral["MS_SNOWPACK_RUNOFF"]
    .sel(time=slice(t1, t2))
    .sum(dim="time")
)



ms_water_nolateral_sum["MS_WATER"] = (
    (ds_nolateral["MS_WATER"]/ds_nolateral["snd"])
    .sel(time=slice(t1, t2))
    .mean(dim="time")
)

ms_water_lateral_sum["MS_WATER"] = (
    (ds_lateral["MS_WATER"]/ds_lateral["snd"])
    .sel(time=slice(t1, t2))
    .mean(dim="time")
)

### Plot difference in cumulative snowpack runoff and average liquid water in snow between t1 and t2

In [ ]:
# Set global font size for all text elements
plt.rcParams.update({
    'font.size': 17,          # Default font size for text
    'axes.titlesize': 17,     # Font size for titles
    'axes.labelsize': 17,     # Font size for x and y labels
    'xtick.labelsize': 17,    # Font size for x-axis tick labels
    'ytick.labelsize': 17,    # Font size for y-axis tick labels
    'legend.fontsize': 17,    # Font size for legend text
})

# Create side-by-side plots
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

# Plot first dataset
var = "MS_SNOWPACK_RUNOFF"
runoff_nolateral_sum[var].plot(ax=ax[0], cbar_kwargs={"label": "Snowpack runoff (kg/m$^2$)"})
ax[0].set_title("snowpack runoff, no lateral flow")
ax[0].set_xlabel("Easting (m)")
ax[0].set_ylabel("Northing (m)")

# Plot second dataset
runoff_lateral_sum[var].plot(ax=ax[1], cbar_kwargs={"label": "Snowpack runoff (kg/m$^2$)"})
ax[1].set_title("snowpack runoff, lateral flow")
ax[1].set_xlabel("Easting (m)")
ax[1].set_ylabel("Northing (m)")

(runoff_lateral_sum[var]-runoff_nolateral_sum[var]).plot(ax=ax[2], cmap='bwr_r', cbar_kwargs={"label": "Snowpack runoff (kg/m$^2$)"})
ax[2].set_title("snowpack runoff (lateral - no lateral)")
ax[2].set_xlabel("Easting (m)")
ax[2].set_ylabel("Northing (m)")

ax[0].text(-0.18, 1.05, '(a)', transform=ax[0].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
ax[1].text(-0.18, 1.05, '(b)', transform=ax[1].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
ax[2].text(-0.18, 1.05, '(c)', transform=ax[2].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')


ax[0].scatter(east1, north1, color="red", s=250, edgecolor="white", zorder=5, linewidth=4)
ax[1].scatter(east1, north1, color="red", s=250, edgecolor="white", zorder=5, linewidth=4)
ax[2].scatter(east1, north1, color="red", s=250, edgecolor="white", zorder=5, linewidth=4)

ax[0].scatter(east2, north2, color="blue", s=250, edgecolor="white", zorder=5, linewidth=4)
ax[1].scatter(east2, north2, color="blue", s=250, edgecolor="white", zorder=5, linewidth=4)
ax[2].scatter(east2, north2, color="blue", s=250, edgecolor="white", zorder=5, linewidth=4)

for a in ax:
    a.text(
        east1, north1+280, "A",
        ha="center", va="center",
        fontsize=18, weight="bold",
        color="red",
        path_effects=[pe.withStroke(linewidth=4, foreground="white")],
        zorder=7
    )

    a.text(
        east2, north2+280, "B",
        ha="center", va="center",
        fontsize=18, weight="bold",
        color="blue",
        path_effects=[pe.withStroke(linewidth=4, foreground="white")],
        zorder=7
    )


plt.tight_layout()
plt.show()



# Create side-by-side plots
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

# Plot first dataset
ms_water_nolateral_sum["MS_WATER"].plot(ax=ax[0], cbar_kwargs={"label": "Average LWC (kg/m$^3$)"})
ax[0].set_title("water content, no lateral flow")
ax[0].set_xlabel("Easting (m)")
ax[0].set_ylabel("Northing (m)")

# Plot second dataset
ms_water_lateral_sum["MS_WATER"].plot(ax=ax[1], cbar_kwargs={"label": "Average LWC (kg/m$^3$)"})
ax[1].set_title("water content, lateral flow")
ax[1].set_xlabel("Easting (m)")
ax[1].set_ylabel("Northing (m)")

(ms_water_lateral_sum["MS_WATER"]-ms_water_nolateral_sum["MS_WATER"]).plot(ax=ax[2], cmap='bwr_r', cbar_kwargs={"label": "Average LWC (kg/m$^3$)"})
ax[2].set_title("water content (lateral - no lateral)")
ax[2].set_xlabel("Easting (m)")
ax[2].set_ylabel("Northing (m)")

ax[0].text(-0.18, 1.05, '(a)', transform=ax[0].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
ax[1].text(-0.18, 1.05, '(b)', transform=ax[1].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
ax[2].text(-0.18, 1.05, '(c)', transform=ax[2].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')

plt.tight_layout()
plt.show()

### Calculate difference, if over the full 

In [ ]:
total_diff = (
    runoff_lateral_sum["MS_SNOWPACK_RUNOFF"]
    - runoff_nolateral_sum["MS_SNOWPACK_RUNOFF"]
).sum(dim=("easting", "northing"))

print(total_diff.values)

In [ ]:
east = east2
north = north2

ts_nolateral = ds_nolateral["MS_SNOWPACK_RUNOFF"].sel(northing=north, easting=east, method="nearest").sel(time=slice(t1, t2)).cumsum(dim="time")
ts_lateral   = ds_lateral["MS_SNOWPACK_RUNOFF"].sel(northing=north, easting=east, method="nearest").sel(time=slice(t1, t2)).cumsum(dim="time")

fig, ax = plt.subplots(figsize=(12, 6))

ts_nolateral.plot(ax=ax, label="No Lateral")
ts_lateral.plot(ax=ax, label="Lateral")

ax.set_title("Snowpack Runoff at Grid Point")
ax.set_xlabel("Time")
ax.set_ylabel("Runoff")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 6))


east = east1
north = north1

ts_nolateral = (
    ds_nolateral["MS_SNOWPACK_RUNOFF"]
    .sel(northing=north, easting=east, method="nearest")
    .sel(time=slice(t1, t2))
    .cumsum(dim="time")
)

ts_lateral = (
    ds_lateral["MS_SNOWPACK_RUNOFF"]
    .sel(northing=north, easting=east, method="nearest")
    .sel(time=slice(t1, t2))
    .cumsum(dim="time")
)

# Difference in cumulative runoff
ts_diff = ts_lateral - ts_nolateral

# Primary axis
ts_nolateral.plot(ax=ax[0], label="No lateral flow", color="tab:blue")
ts_lateral.plot(ax=ax[0], label="Lateral flow", color="tab:orange")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Cumulative Runoff")
ax[0].set_title("Snowpack Runoff at Grid Point A")

# Secondary axis
ax2 = ax[0].twinx()
ts_diff.plot(ax=ax2, label="Difference",
             color="tab:purple", linestyle="--")
ax2.set_title("")
ax2.set_ylabel("Cumulative Runoff Difference (kg/m$^2$)")

# Combine legends from both axes
lines1, labels1 = ax[0].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax[0].legend(lines1 + lines2, labels1 + labels2, loc="best")


east = east2
north = north2

ts_nolateral = (
    ds_nolateral["MS_SNOWPACK_RUNOFF"]
    .sel(northing=north, easting=east, method="nearest")
    .sel(time=slice(t1, t2))
    .cumsum(dim="time")
)

ts_lateral = (
    ds_lateral["MS_SNOWPACK_RUNOFF"]
    .sel(northing=north, easting=east, method="nearest")
    .sel(time=slice(t1, t2))
    .cumsum(dim="time")
)

# Difference in cumulative runoff
ts_diff = ts_lateral - ts_nolateral

# Primary axis
ts_nolateral.plot(ax=ax[1], label="No lateral flow", color="tab:blue")
ts_lateral.plot(ax=ax[1], label="Lateral flow", color="tab:orange")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Cumulative Runoff")
ax[1].set_title("Snowpack Runoff at Grid Point B")

# Secondary axis
ax2 = ax[1].twinx()
ts_diff.plot(ax=ax2, label="Difference", 
             color="tab:purple", linestyle="--")
ax2.set_title("")
ax2.set_ylabel("Cumulative Runoff Difference (kg/m$^2$)")

# Combine legends from both axes
lines1, labels1 = ax[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax[1].legend(lines1 + lines2, labels1 + labels2, loc="best")


# Add labels
ax[0].text(-0.18, 1.05, '(a)', transform=ax[0].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')
ax[1].text(-0.18, 1.05, '(b)', transform=ax[1].transAxes, fontsize=18, fontweight='bold', va='top', ha='right')


plt.tight_layout()
plt.show()